# Previsão de Avaliações de Filmes com Regressão Ridge e Árvore de Decisão

**Dataset:** MovieLens *ml-latest-small* &nbsp;·&nbsp; **Tarefa:** regressão supervisionada &nbsp;·&nbsp; **Métricas:** MAE e RMSE

## Resumo

Este projeto prevê a avaliação (em estrelas) que um utilizador atribuiria a um filme, com base no identificador do utilizador, no identificador do filme e nos géneros cinematográficos. Comparam-se dois modelos, **Regressão Ridge** e **Árvore de Decisão**, com otimização de hiperparâmetros por validação cruzada e avaliação final num conjunto de teste independente. Os resultados são comparados com três referências simples (média global, média por utilizador e média por filme).

## Índice

0. Configuração
1. Carregamento e exploração dos dados
2. Junção das avaliações aos títulos e géneros
3. Divisão treino/teste
4. Representação das variáveis
5. Definição dos modelos
6. Validação cruzada e otimização
7. Avaliação final no conjunto de teste
8. Interpretação do modelo Ridge
9. Conclusões e limitações

**Requisitos:** Python 3.9+, `pandas`, `numpy`, `scikit-learn` e `matplotlib`. Os ficheiros `ratings.csv` e `movies.csv` (MovieLens *ml-latest-small*) devem estar na pasta do notebook, ou o caminho deve ser ajustado em `PASTA_DADOS`.

## 0. Configuração

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

# Parâmetros globais
SEMENTE = 42                       # reprodutibilidade
PASTA_DADOS = Path(".")            # pasta com ratings.csv e movies.csv
NOTA_MIN, NOTA_MAX = 0.5, 5.0      # escala de avaliação do MovieLens

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

## 1. Carregamento e exploração dos dados

Os dados estão divididos em dois ficheiros: `ratings.csv` (uma linha por avaliação) e `movies.csv` (título e géneros de cada filme).

In [ ]:
for nome in ("ratings.csv", "movies.csv"):
    if not (PASTA_DADOS / nome).exists():
        raise FileNotFoundError(
            f"Ficheiro '{nome}' não encontrado em '{PASTA_DADOS.resolve()}'. "
            "Ajuste PASTA_DADOS ou copie os ficheiros do MovieLens para essa pasta."
        )

ratings = pd.read_csv(PASTA_DADOS / "ratings.csv")
movies = pd.read_csv(PASTA_DADOS / "movies.csv")

display(ratings.head())
display(movies.head())

In [ ]:
n_aval, n_users, n_filmes = len(ratings), ratings["userId"].nunique(), len(movies)
n_filmes_avaliados = ratings["movieId"].nunique()

print(f"Número de avaliações:                {n_aval:,}")
print(f"Número de utilizadores:              {n_users:,}")
print(f"Número de filmes no catálogo:        {n_filmes:,}")
print(f"Filmes com pelo menos 1 avaliação:   {n_filmes_avaliados:,}")
print(f"Densidade da matriz utilizador×filme: {n_aval / (n_users * n_filmes_avaliados):.2%}")
print("\nValores em falta nas avaliações:")
print(ratings.isna().sum())
print(f"\nAvaliações duplicadas (utilizador, filme): {ratings.duplicated(['userId', 'movieId']).sum()}")

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 3.8))

contagem = ratings["rating"].value_counts().sort_index()
eixos[0].bar(contagem.index.astype(str), contagem.values, color="#4C72B0")
eixos[0].set(title="Distribuição das notas", xlabel="Nota", ylabel="Nº de avaliações")

por_user = ratings.groupby("userId").size()
eixos[1].hist(por_user, bins=40, color="#55A868")
eixos[1].set(title="Avaliações por utilizador", xlabel="Nº de avaliações", ylabel="Nº de utilizadores")

por_filme = ratings.groupby("movieId").size()
eixos[2].hist(por_filme, bins=40, color="#C44E52", log=True)
eixos[2].set(title="Avaliações por filme (escala log)", xlabel="Nº de avaliações", ylabel="Nº de filmes")

plt.tight_layout()
plt.show()

print(f"Nota média global: {ratings['rating'].mean():.3f}")
print(f"Mediana de avaliações por utilizador: {por_user.median():.0f} | por filme: {por_filme.median():.0f}")

A distribuição das notas é assimétrica, com predomínio de notas altas, e a atividade é muito desigual: poucos utilizadores e poucos filmes concentram grande parte das avaliações. Isto justifica a utilização de modelos que captem o **enviesamento individual** de cada utilizador e de cada filme, e a atenção ao problema de filmes pouco avaliados (secção 3).

## 2. Junção das avaliações aos títulos e géneros

A tabela de avaliações é enriquecida com o título e os géneros de cada filme. A opção `validate="many_to_one"` garante que cada `movieId` corresponde a um único filme.

In [ ]:
avaliacoes = ratings.merge(movies, on="movieId", how="left", validate="many_to_one")

assert avaliacoes["title"].notna().all(), "Existem avaliações sem filme correspondente."
assert len(avaliacoes) == len(ratings)

display(avaliacoes[["userId", "title", "genres", "rating"]].head(10))

*Nota: `.head(n)` devolve as primeiras `n` linhas do DataFrame. As linhas acima pertencem ao utilizador 1.*

**Exemplo: filmes avaliados pelo utilizador 1**

In [ ]:
avaliacoes_user1 = avaliacoes.loc[avaliacoes["userId"] == 1, ["title", "genres", "rating"]]

print(f"Número de filmes avaliados pelo utilizador 1: {len(avaliacoes_user1)}")
display(avaliacoes_user1.head(10))

## 3. Divisão treino/teste

**Objetivo:** dado um utilizador, um filme e os respetivos géneros, prever a nota atribuída.

Os identificadores são convertidos para *string* para serem tratados como categorias nominais e não como quantidades. Reserva-se 80% dos dados para treino e 20% para teste. O conjunto de teste só é utilizado na avaliação final (secção 7).

In [ ]:
X = avaliacoes[["userId", "movieId", "genres"]].copy()
X[["userId", "movieId"]] = X[["userId", "movieId"]].astype(str)
y = avaliacoes["rating"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=SEMENTE
)

print(f"Avaliações de treino: {len(X_treino):,}")
print(f"Avaliações de teste:  {len(X_teste):,}")

### Arranque a frio (*cold start*)

Numa divisão aleatória, alguns utilizadores ou filmes do teste podem não ter qualquer avaliação no treino. Com `handle_unknown="ignore"`, o `OneHotEncoder` representa essas categorias como um vetor nulo, pelo que o modelo recorre apenas ao intercepto e aos géneros. Quantifica-se de seguida a dimensão deste efeito.

In [ ]:
filme_novo = ~X_teste["movieId"].isin(X_treino["movieId"])
user_novo = ~X_teste["userId"].isin(X_treino["userId"])

print(f"Avaliações de teste com filme não visto no treino:      {filme_novo.mean():.2%} ({filme_novo.sum():,})")
print(f"Avaliações de teste com utilizador não visto no treino: {user_novo.mean():.2%} ({user_novo.sum():,})")

## 4. Representação das variáveis

Os identificadores (`userId`, `movieId`) são categorias nominais e os géneros formam uma lista multi-etiqueta separada por `|`. Ambos são convertidos em colunas binárias (0/1) antes de serem utilizados pelos modelos.

**Exemplo ilustrativo**

| userId | movieId | genres |
|---|---|---|
| 1 | 10 | Action\|Thriller |
| 2 | 20 | Comedy |

Depois da codificação binária:

| É o utilizador 1? | É o utilizador 2? | É o filme 10? | É o filme 20? | Action | Comedy | Thriller |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 1 | 0 | 1 | 0 | 1 | 0 | 1 |
| 0 | 1 | 0 | 1 | 0 | 1 | 0 |

Esta codificação evita que o modelo interprete identificadores como grandezas numéricas e permite que os modelos baseados em árvores criem regras por género.

O pré-processador é definido numa única função, reutilizada por todos os modelos, de modo a garantir um tratamento idêntico.

In [ ]:
def criar_preprocessador():
    # One-hot para os identificadores (categorias desconhecidas ignoradas) e
    # indicadores binários para cada género (lista separada por '|').
    return ColumnTransformer([
        ("ids", OneHotEncoder(handle_unknown="ignore"), ["userId", "movieId"]),
        ("generos", CountVectorizer(token_pattern=r"[^|]+", lowercase=False, binary=True), "genres"),
    ])

**Demonstração com cinco registos do conjunto de treino**

In [ ]:
exemplo = X_treino.head(5).copy()
codificador_exemplo = criar_preprocessador()
matriz_exemplo = codificador_exemplo.fit_transform(exemplo)

tabela_exemplo = pd.DataFrame(
    matriz_exemplo.toarray(),
    columns=codificador_exemplo.get_feature_names_out(),
    index=exemplo.index,
)

print("Dados antes da codificação:")
display(exemplo)
print("Depois (one-hot dos identificadores + indicadores de género):")
display(tabela_exemplo)

## 5. Definição dos modelos

### 5.1 Regressão Ridge

A Regressão Ridge é uma regressão linear com regularização L2: além de minimizar o erro das previsões, penaliza coeficientes elevados, o que reduz o sobreajuste (*overfitting*). Com esta representação, o modelo é aditivo:

> **Nota prevista = intercepto + efeito do utilizador + efeito do filme + efeito dos géneros**

O efeito do utilizador capta a tendência para atribuir notas mais altas ou mais baixas, o efeito do filme capta a sua qualidade média e o efeito dos géneros capta o contributo médio de cada género. O hiperparâmetro `alpha` controla a intensidade da regularização e é otimizado na secção 6.

In [ ]:
modelo_ridge = Pipeline([
    ("categorias", criar_preprocessador()),
    ("regressao", Ridge(alpha=1.0, solver="lsqr")),   # lsqr: eficiente para matrizes esparsas
])
display(modelo_ridge)

### 5.2 Árvore de Decisão

A Árvore de Decisão divide os dados através de regras sucessivas, com o objetivo de formar grupos com avaliações semelhantes. A complexidade é controlada por dois hiperparâmetros, otimizados na secção 6:

- `max_depth`: número máximo de níveis de decisão consecutivos;
- `min_samples_leaf`: número mínimo de avaliações de treino em cada folha.

In [ ]:
modelo_arvore = Pipeline([
    ("categorias", criar_preprocessador()),
    ("regressao", DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=SEMENTE)),
])
display(modelo_arvore)

### 5.3 Prevenção de fuga de dados (*data leakage*)

Cada modelo integra a codificação e a regressão numa `Pipeline`. Em cada divisão da validação cruzada, as categorias são aprendidas apenas com a parte de treino, e a parte de validação limita-se a aplicar essa codificação. O procedimento de preparação é assim idêntico no treino e na previsão, sem fuga de informação.

## 6. Validação cruzada e otimização

Utiliza-se validação cruzada com três divisões (*3-fold*) sobre o conjunto de treino: em cada iteração, duas partes treinam e uma valida, até que todas tenham sido usadas na validação. A métrica de seleção é o erro quadrático médio (MSE), reportado como RMSE para ficar na escala das estrelas.

O `GridSearchCV` testa exaustivamente todas as combinações de hiperparâmetros candidatas.

In [ ]:
validacao = KFold(n_splits=3, shuffle=True, random_state=SEMENTE)


def rmse_da_busca(busca):
    # Converte o melhor score (MSE negativo) em RMSE.
    return np.sqrt(-busca.best_score_)


def extremos_da_grelha(busca, grelha):
    # Devolve os hiperparâmetros cujo melhor valor está num extremo da grelha.
    # Isso indica que o ótimo pode estar fora do intervalo testado.
    alerta = []
    for nome, valores in grelha.items():
        melhor = busca.best_params_[nome]
        if len(valores) > 1 and melhor in (min(valores), max(valores)):
            alerta.append(f"{nome} = {melhor}")
    return alerta

### 6.1 Ridge: otimização de `alpha`

Valores elevados de `alpha` restringem mais o modelo, o que reduz o sobreajuste mas, se for excessivo, pode causar subajuste (*underfitting*).

In [ ]:
grelha_ridge = {"regressao__alpha": [0.001, 0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]}

busca_ridge = GridSearchCV(
    estimator=modelo_ridge,
    param_grid=grelha_ridge,
    scoring="neg_mean_squared_error",
    cv=validacao,
    n_jobs=-1,
)
busca_ridge.fit(X_treino, y_treino)

rmse_validacao_ridge = rmse_da_busca(busca_ridge)
print("Melhores parâmetros:", busca_ridge.best_params_)
print(f"RMSE de validação:  {rmse_validacao_ridge:.3f}")

alerta = extremos_da_grelha(busca_ridge, grelha_ridge)
if alerta:
    print("Atenção: ótimo num extremo da grelha ->", ", ".join(alerta))
else:
    print("O ótimo situa-se no interior da grelha.")

### 6.2 Árvore de Decisão: otimização de `max_depth` e `min_samples_leaf`

A grelha cobre profundidades de 10 a 50 e mínimos de 5 a 50 avaliações por folha. Esta pesquisa é mais lenta do que a do Ridge, porque ajusta 16 combinações em 3 divisões.

In [ ]:
grelha_arvore = {
    "regressao__max_depth": [10, 20, 30, 50],
    "regressao__min_samples_leaf": [5, 10, 20, 50],
}

busca_arvore = GridSearchCV(
    estimator=modelo_arvore,
    param_grid=grelha_arvore,
    scoring="neg_mean_squared_error",
    cv=validacao,
    n_jobs=-1,
)
busca_arvore.fit(X_treino, y_treino)

rmse_validacao_arvore = rmse_da_busca(busca_arvore)
print("Melhores parâmetros:", busca_arvore.best_params_)
print(f"RMSE de validação:  {rmse_validacao_arvore:.3f}")

alerta = extremos_da_grelha(busca_arvore, grelha_arvore)
if alerta:
    print("Atenção: ótimo num extremo da grelha ->", ", ".join(alerta))
else:
    print("O ótimo situa-se no interior da grelha.")

### 6.3 Comparação em validação

In [ ]:
comparacao_validacao = pd.DataFrame([
    {"Modelo": "Ridge", "RMSE de validação": rmse_validacao_ridge,
     "Melhores parâmetros": busca_ridge.best_params_},
    {"Modelo": "Árvore de Decisão", "RMSE de validação": rmse_validacao_arvore,
     "Melhores parâmetros": busca_arvore.best_params_},
]).sort_values("RMSE de validação").reset_index(drop=True)

display(comparacao_validacao)

vencedor_validacao = comparacao_validacao.loc[0, "Modelo"]
print(f"\nModelo com menor erro de validação: {vencedor_validacao}")

O RMSE está expresso na escala das avaliações (estrelas) e penaliza mais os erros elevados. **Não** representa uma percentagem de acerto.

## 7. Avaliação final no conjunto de teste

Os modelos otimizados são avaliados no conjunto de teste, que não participou no treino nem na escolha dos hiperparâmetros. Os resultados são comparados com três referências simples, calculadas apenas com o conjunto de treino:

1. **Média global:** prevê sempre a média das avaliações de treino;
2. **Média por utilizador:** média das notas do próprio utilizador;
3. **Média por filme:** média das notas do próprio filme.

Nas referências 2 e 3, utilizadores ou filmes sem histórico recebem a média global.

Como a regressão linear pode produzir valores fora da escala, avalia-se também o Ridge com previsões limitadas ao intervalo [0,5; 5,0].

In [ ]:
melhor_ridge = busca_ridge.best_estimator_
melhor_arvore = busca_arvore.best_estimator_

# Referências (calculadas apenas com o treino)
media_global = y_treino.mean()
media_por_user = y_treino.groupby(X_treino["userId"]).mean()
media_por_filme = y_treino.groupby(X_treino["movieId"]).mean()

# Previsões dos modelos
prev_ridge = melhor_ridge.predict(X_teste)

previsoes = {
    "Referência: média global": np.full(len(y_teste), media_global),
    "Referência: média por utilizador": X_teste["userId"].map(media_por_user).fillna(media_global).to_numpy(),
    "Referência: média por filme": X_teste["movieId"].map(media_por_filme).fillna(media_global).to_numpy(),
    "Árvore de Decisão otimizada": melhor_arvore.predict(X_teste),
    "Ridge otimizada": prev_ridge,
    "Ridge otimizada (limitada a [0,5; 5,0])": np.clip(prev_ridge, NOTA_MIN, NOTA_MAX),
}

print(f"Média das avaliações de treino: {media_global:.3f}")
print(f"Número de previsões no teste:   {len(y_teste):,}")
print(f"Previsões do Ridge fora de [0,5; 5,0]: {((prev_ridge < NOTA_MIN) | (prev_ridge > NOTA_MAX)).mean():.2%}")

### 7.1 Métricas

In [ ]:
resultados = pd.DataFrame([
    {
        "Modelo": nome,
        "MAE": mean_absolute_error(y_teste, p),
        "RMSE": np.sqrt(mean_squared_error(y_teste, p)),
    }
    for nome, p in previsoes.items()
])

rmse_ref = resultados.loc[0, "RMSE"]
mae_ref = resultados.loc[0, "MAE"]
resultados["Melhoria RMSE vs média global"] = 1 - resultados["RMSE"] / rmse_ref
resultados["Melhoria MAE vs média global"] = 1 - resultados["MAE"] / mae_ref

tabela_resultados = resultados.sort_values("RMSE").reset_index(drop=True)
display(
    tabela_resultados.style
    .format({"MAE": "{:.4f}", "RMSE": "{:.4f}",
             "Melhoria RMSE vs média global": "{:.1%}",
             "Melhoria MAE vs média global": "{:.1%}"})
    .hide(axis="index")
)

- **MAE:** média dos erros absolutos entre as notas reais e previstas. Um MAE de 0,67, por exemplo, significa que o modelo erra, em média, 0,67 estrelas.
- **RMSE:** raiz do erro quadrático médio, que atribui maior peso aos erros grandes.
- Para ambas as métricas, valores menores indicam melhores previsões.

### 7.2 Notas reais e previstas

In [ ]:
previsao_final = previsoes["Ridge otimizada (limitada a [0,5; 5,0])"]

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.2))

# Nota média prevista para cada nota real
media_prevista = pd.Series(previsao_final).groupby(y_teste.to_numpy()).mean()
eixos[0].plot([NOTA_MIN, NOTA_MAX], [NOTA_MIN, NOTA_MAX], "--", color="grey", label="Previsão perfeita")
eixos[0].plot(media_prevista.index, media_prevista.values, "o-", color="#4C72B0", label="Ridge (média prevista)")
eixos[0].set(title="Nota prevista média por nota real", xlabel="Nota real", ylabel="Nota prevista média")
eixos[0].legend()

# Distribuição dos erros
erros = y_teste.to_numpy() - previsao_final
eixos[1].hist(erros, bins=40, color="#55A868")
eixos[1].axvline(0, color="black", lw=1)
eixos[1].set(title="Distribuição dos erros (real − previsto)", xlabel="Erro (estrelas)", ylabel="Nº de avaliações")

plt.tight_layout()
plt.show()

O gráfico da esquerda mostra a tendência típica de modelos de regressão: as notas extremas (muito baixas ou muito altas) são previstas de forma "encolhida" em direção à média, porque o modelo minimiza o erro quadrático médio.

Seguem-se exemplos concretos de previsões do Ridge, com os dez primeiros registos do teste e as dez maiores falhas.

In [ ]:
exemplos_teste = X_teste.copy()
exemplos_teste["Nota real"] = y_teste.to_numpy()
exemplos_teste["Nota prevista"] = previsao_final
exemplos_teste["Erro absoluto"] = (exemplos_teste["Nota real"] - exemplos_teste["Nota prevista"]).abs()

titulos = movies[["movieId", "title"]].astype({"movieId": str})
exemplos_teste = exemplos_teste.merge(titulos, on="movieId", how="left", validate="many_to_one")

colunas = ["userId", "title", "genres", "Nota real", "Nota prevista", "Erro absoluto"]

print("Primeiros registos do conjunto de teste:")
display(exemplos_teste[colunas].head(10).round(3))

print("Dez maiores erros absolutos:")
display(exemplos_teste.nlargest(10, "Erro absoluto")[colunas].round(3))

## 8. Interpretação do modelo Ridge

Por ser um modelo linear, os coeficientes do Ridge têm leitura direta: representam o desvio (em estrelas) face ao intercepto associado a cada género, utilizador ou filme. Os efeitos dos géneros são os mais fáceis de interpretar, por serem poucos e partilhados por muitos filmes.

In [ ]:
nomes = melhor_ridge.named_steps["categorias"].get_feature_names_out()
coefs = pd.Series(melhor_ridge.named_steps["regressao"].coef_, index=nomes)

efeito_generos = (
    coefs[coefs.index.str.startswith("generos__")]
    .rename(lambda n: n.replace("generos__", ""))
    .sort_values()
)

fig, ax = plt.subplots(figsize=(7, 6))
cores = np.where(efeito_generos.values >= 0, "#55A868", "#C44E52")
ax.barh(efeito_generos.index, efeito_generos.values, color=cores)
ax.axvline(0, color="black", lw=0.8)
ax.set(title="Efeito de cada género na nota prevista (coeficiente Ridge)", xlabel="Coeficiente (estrelas)")
plt.tight_layout()
plt.show()

print(f"Intercepto: {melhor_ridge.named_steps['regressao'].intercept_:.3f}")

Os coeficientes de género são **efeitos parciais**: medem o contributo de cada género depois de descontados os efeitos do utilizador e do filme, pelo que não devem ser lidos como "a nota média do género". Um filme com vários géneros acumula a soma dos respetivos efeitos.

## 9. Conclusões e limitações

In [ ]:
melhor = tabela_resultados.loc[~tabela_resultados["Modelo"].str.startswith("Referência")].iloc[0]
melhor_ref = tabela_resultados.loc[tabela_resultados["Modelo"].str.startswith("Referência")].iloc[0]
ridge_r = tabela_resultados.set_index("Modelo").loc["Ridge otimizada"]
arvore_r = tabela_resultados.set_index("Modelo").loc["Árvore de Decisão otimizada"]

print(f"Melhor modelo no teste: {melhor['Modelo']} (RMSE = {melhor['RMSE']:.3f}; MAE = {melhor['MAE']:.3f})")
print(f"Melhor referência:      {melhor_ref['Modelo']} (RMSE = {melhor_ref['RMSE']:.3f}; MAE = {melhor_ref['MAE']:.3f})")
print(f"Redução do RMSE face à média global: Ridge {ridge_r['Melhoria RMSE vs média global']:.1%} | "
      f"Árvore {arvore_r['Melhoria RMSE vs média global']:.1%}")
print(f"RMSE validação -> teste (Ridge): {rmse_validacao_ridge:.3f} -> {ridge_r['RMSE']:.3f}")
print(f"RMSE validação -> teste (Árvore): {rmse_validacao_arvore:.3f} -> {arvore_r['RMSE']:.3f}")

### Conclusões

1. **Desempenho:** os modelos foram comparados com referências simples e avaliados num conjunto de teste independente. Os valores finais, e a comparação entre o Ridge, a Árvore e as referências por utilizador e por filme, estão na tabela da secção 7.1 e no resumo acima.
2. **Adequação do modelo aditivo:** a estrutura "utilizador + filme + géneros" capta dois fenómenos centrais neste problema, o enviesamento de cada utilizador e a qualidade média de cada filme. É por isso que o Ridge se compara favoravelmente com um modelo que apenas conhece a média global.
3. **Limitação da Árvore de Decisão:** com milhares de colunas *one-hot* esparsas, cada divisão tende a isolar um único utilizador ou filme, o que limita a partilha de informação entre avaliações. O Ridge, ao contrário, soma naturalmente os efeitos aditivos.
4. **Generalização:** a proximidade entre o RMSE de validação e o RMSE de teste (ver resumo acima) indica que o processo de seleção de hiperparâmetros não produziu um sobreajuste ao conjunto de validação.
5. **Escala das previsões:** limitar as previsões ao intervalo [0,5; 5,0] é uma correção simples e sem custos, que nunca aumenta o erro, porque as notas reais estão sempre nesse intervalo.

### Limitações

- **Divisão aleatória:** não reflete o uso real, em que se prevêem avaliações futuras a partir de avaliações passadas. Uma **divisão temporal** (usando `timestamp`) seria mais realista e tenderia a produzir erros mais elevados.
- **Arranque a frio:** utilizadores e filmes sem histórico no treino recebem apenas o intercepto e o efeito dos géneros (secção 3).
- **Variáveis limitadas:** não são usados o ano do filme, o número de avaliações, as *tags* nem a data da avaliação.
- **Hiperparâmetros:** a pesquisa em grelha é discreta e limitada; pesquisas mais finas ou aleatórias poderiam melhorar ligeiramente os resultados.
- **Modelos mais fortes disponíveis:** o problema é clássico em sistemas de recomendação e é geralmente bem servido por fatorização matricial.

### Trabalho futuro

- Avaliação com divisão temporal e com validação cruzada por utilizador.
- Fatorização matricial (SVD, ALS) e modelos com interações utilizador×género.
- Variáveis adicionais (ano de lançamento, popularidade, *tags*) e modelos de *gradient boosting*.
- Métricas orientadas ao *ranking* (por exemplo, NDCG@K), mais próximas do objetivo de recomendar filmes.